## Building A Chatbot
In this video We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This video tutorial will cover the basics which will be helpful for those two more advanced topics.

In [9]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")



In [3]:
from langchain_groq import ChatGroq
model=ChatGroq(model="llama-3.1-8b-instant",groq_api_key=groq_api_key)
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x000001C46AF8F010>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001C46AF8EF20>, model_name='llama-3.1-8b-instant', groq_api_key=SecretStr('**********'))

In [5]:
from langchain_core.messages import HumanMessage
response=model.invoke([HumanMessage(content="Hi,My name is Vikash and i am a Data Analyst")])
response  

AIMessage(content='Nice to meet you, Vikash. As a Data Analyst, you must be working with numbers, statistics, and data visualization to help organizations make informed decisions. What kind of projects or industries are you working in, and what tools or technologies are you most familiar with?', response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 48, 'total_tokens': 103, 'completion_time': 0.081041291, 'completion_tokens_details': None, 'prompt_time': 0.00232217, 'prompt_tokens_details': None, 'queue_time': 0.04538604, 'total_time': 0.083363461}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-b2c01629-28a1-4ab2-b4f8-fb07d239167f-0')

In [8]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi ,My name is Vikash and i am a Data Analyst"),
        AIMessage(content="Nice to meet you, Vikash. As a Data Analyst, you must be working with numbers, statistics, and data visualization to help organizations make informed decisions. What kind of projects or industries are you working in, and what tools or technologies are you most familiar with?"),
        HumanMessage(content = "hey Whats my name and what do i do ? ")
    ]
)


AIMessage(content="Your name is Vikash, and you're a Data Analyst.", response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 123, 'total_tokens': 137, 'completion_time': 0.015944607, 'completion_tokens_details': None, 'prompt_time': 0.007720743, 'prompt_tokens_details': None, 'queue_time': 0.045857077, 'total_time': 0.02366535}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-41a38378-c42b-4bbc-b8ef-34eedc055e68-0')

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [10]:
%pip install langchain_community

Note: you may need to restart the kernel to use updated packages.


c:\Users\VikashMahato\GenAI_Learning\venv\Scripts\python.exe: No module named pip


In [12]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory


store ={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [13]:
config = ({"configurable": {"session_id": "chat_1"}})

In [14]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi ,My name is Vikash and i am a Data Analyst")],
    config=config
)

In [15]:
response.content

"Hello Vikash. It's nice to meet you. As a Data Analyst, I'm sure you're skilled in working with data to uncover insights and inform business decisions. What kind of projects or areas are you currently working on or interested in?"

In [16]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config
)

AIMessage(content='Your name is Vikash.', response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 112, 'total_tokens': 119, 'completion_time': 0.005608345, 'completion_tokens_details': None, 'prompt_time': 0.007007807, 'prompt_tokens_details': None, 'queue_time': 0.045152873, 'total_time': 0.012616152}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-70e0375d-a02d-41ed-8a84-c28787beb60a-0')

In [17]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain information about individual users or their personal details. Each time you interact with me, it's a new conversation and I don't have any prior knowledge about you. If you'd like to share your name with me, I'd be happy to chat with you!"

In [18]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is John")],
    config=config1
)
response.content

"Nice to meet you, John. I'm glad you shared your name with me. Now that we've made contact, we can have a more personalized conversation. What's on your mind today? Would you like to talk about something specific or just have a casual chat?"

In [19]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is John.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [21]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder 
prompt = ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Answer all question to the best of your ability."),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain = prompt|model

In [23]:
chain.invoke({"messages": [HumanMessage(content = "Hi ,My name is Vikash ")]})


AIMessage(content="Nice to meet you, Vikash. I'm happy to assist you with any questions or topics you'd like to discuss. Is there something specific on your mind, or would you like to start a conversation?", response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 59, 'total_tokens': 102, 'completion_time': 0.054809312, 'completion_tokens_details': None, 'prompt_time': 0.003094959, 'prompt_tokens_details': None, 'queue_time': 0.04584582, 'total_time': 0.057904271}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'finish_reason': 'stop', 'logprobs': None}, id='run-7175db6c-3fd5-47e5-8f81-91c11e3ac56a-0')

In [24]:
with_message_history = RunnableWithMessageHistory(chain, get_session_history)

In [26]:
config = ({"configurable": {"session_id": "chat3"}})
response=with_message_history.invoke(
    [HumanMessage(content="Hi ,My name is Vikash")],
    config=config 
)
response

AIMessage(content="Nice to meet you, Vikash. I'm here to help with any questions or topics you'd like to discuss. How's your day going so far?", response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 58, 'total_tokens': 91, 'completion_time': 0.034545682, 'completion_tokens_details': None, 'prompt_time': 0.002878174, 'prompt_tokens_details': None, 'queue_time': 0.047055116, 'total_time': 0.037423856}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_f757f4b0bf', 'finish_reason': 'stop', 'logprobs': None}, id='run-67bd909a-0b6c-45f1-8c61-66280f34fd67-0')

In [27]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [30]:
response = chain.invoke({"messages":[HumanMessage(content ="Hi My name is Vikash")], 
                         "language": "Hindi"}
                         )
response.content

'नमस्ते विकास, मैं आपकी सहायता के लिए यहाँ हूँ। क्या आपके पास कोई प्रश्न या समस्या है जिसमें मैं आपकी मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [33]:
with_message_history = RunnableWithMessageHistory(
    chain, 
    get_session_history,
    input_messages_key="messages" 
     )


In [35]:
config = ({"configurable": {"session_id": "chat4"}})
response=with_message_history.invoke(
    {"messages":[HumanMessage(content="Hi My name is Vikash")],"language": "Hindi"},
    config = config
)
response.content

'नमस्ते विकाश जी, दोबारा आपका स्वागत है! क्या आपको कोई विशेष चीज़ पता करनी है, या मैं आपकी किसी समस्या का समाधान करने में मदद कर सकता हूँ?'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.

'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [36]:
from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm bob"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

c:\Users\VikashMahato\GenAI_Learning\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]
c:\Users\VikashMahato\GenAI_Learning\venv\lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\VikashMahato\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` 

[SystemMessage(content="you're a good assistant"),
 HumanMessage(content='I like vanilla ice cream'),
 AIMessage(content='nice'),
 HumanMessage(content='whats 2 + 2'),
 AIMessage(content='4'),
 HumanMessage(content='thanks'),
 AIMessage(content='no problem!'),
 HumanMessage(content='having fun?'),
 AIMessage(content='yes!')]

In [39]:
from operator import itemgetter 
from langchain_core.runnables import RunnablePassthrough

chain = (
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model 


)

response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What icecream do  i like?")],
    "language":"English"
    }
)

response.content

"Unfortunately, I don't have that information. You didn't tell me what your favorite ice cream flavor is. Would you like to share?"

In [40]:
response = chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What math problem did i ask?")],
    "language":"English"
    }
)

response.content

'You asked me a simple math problem: 2 + 2.'

In [41]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [42]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

"I don't know your name, we just started talking. I'm here to help, though!"

In [43]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

"You didn't ask a math problem yet. Our conversation just started, and we only discussed your name. If you'd like to ask a math question, I'll do my best to help you solve it."